In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import shutil
import os
from IPython.display import FileLink, display

# 1. Dosyanın Kaggle içindeki adresi
kaynak_dosya = "/kaggle/input/mimii-mini-dataset-100-nums-of-all/mimii-mini/-6_dB_fan/fan/id_00/abnormal/00000005.wav"

# 2. İndirilecek dosyanın adı ne olsun? (Output klasörüne kopyalıyoruz)
hedef_dosya = "anormal.wav"

# 3. Dosyayı kopyala
shutil.copy(kaynak_dosya, hedef_dosya)

# 4. İndirme Linkini Göster
print("Dosya hazır! Bilgisayarına indirmek için aşağıdaki mavi linke tıkla: 👇")
display(FileLink(hedef_dosya))

Dosya hazır! Bilgisayarına indirmek için aşağıdaki mavi linke tıkla: 👇


/kaggle/working/anormal.wav

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchaudio
from torchaudio import transforms
import matplotlib.pyplot as plt
import glob
import random

# ==========================================
# ⚙️ EĞİTİM AYARLARI
# ==========================================
DATA_PATH = "/kaggle/input/mimii-mini-dataset-100-nums-of-all/mimii-mini/-6_dB_fan" 

SAMPLE_RATE = 16000
EPOCHS = 20           # Eğitimin ne kadar süreceği (20 ideal)
BATCH_SIZE = 16       # Her seferde kaç dosya alacağı
LEARNING_RATE = 0.001
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"🚀 Eğitim {DEVICE} üzerinde yapılacak.")

# ==========================================
# 1. VERİ SETİ SINIFI (DATASET)
# ==========================================
class FanDataset(Dataset):
    def __init__(self, root_dir):
        self.files = []
        self.labels = []
        
        # Dosyaları Bul
        # Normal Sesler (Etiket: 0)
        normal_files = glob.glob(os.path.join(root_dir, "fan", "*", "normal", "*.wav"))
        self.files.extend(normal_files)
        self.labels.extend([0] * len(normal_files)) # 0 = Normal
        
        # Arızalı Sesler (Etiket: 1)
        abnormal_files = glob.glob(os.path.join(root_dir, "fan", "*", "abnormal", "*.wav"))
        self.files.extend(abnormal_files)
        self.labels.extend([1] * len(abnormal_files)) # 1 = Arıza
        
        # Karıştır (Shuffle)
        combined = list(zip(self.files, self.labels))
        random.shuffle(combined)
        self.files, self.labels = zip(*combined)
        
        print(f"✅ Toplam Dosya: {len(self.files)}")
        print(f"   - Normal: {len(normal_files)}")
        print(f"   - Arıza: {len(abnormal_files)}")

        # Ses İşleme Araçları
        self.mel_transform = transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE, n_mels=128, n_fft=1024, hop_length=512
        )
        self.db_transform = transforms.AmplitudeToDB()

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        label = self.labels[idx]
        
        # 1. Dosyayı Yükle
        waveform, sr = torchaudio.load(path)
        
        
        if waveform.shape[0] > 1:
            waveform = waveform[0:1, :] # Boyut [8, Zaman] -> [1, Zaman] olur
            
        # 2. Resample (16000 Hz)
        if sr != SAMPLE_RATE:
            resampler = transforms.Resample(orig_freq=sr, new_freq=SAMPLE_RATE)
            waveform = resampler(waveform)
            
        # 3. 3 Saniyelik Kesit Al 
        num_samples = SAMPLE_RATE * 3
        if waveform.shape[1] > num_samples:
            waveform = waveform[:, :num_samples]
        else:
            padding = num_samples - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, padding))
            
        # 4. Spektrograma Çevir
        spec = self.mel_transform(waveform) # Çıktı şu an: [1, 128, Zaman]
        spec = self.db_transform(spec)
        
        
       
        spec = spec.unsqueeze(0) # [1, 1, 128, Zaman]
        
        #  Resize (128x128)
        spec = torch.nn.functional.interpolate(spec, size=(128, 128), mode='bilinear', align_corners=False)
        
        
        spec = spec.squeeze(0)   # [1, 128, 128]
        
        # 5. Normalize (0-1 arası)
        spec = (spec - spec.min()) / (spec.max() - spec.min() + 1e-6)
        
        return spec, torch.tensor(label, dtype=torch.float32)

# ==========================================
# 2. MODEL MİMARİSİ 
# ==========================================
class FanModel(nn.Module):
    def __init__(self):
        super(FanModel, self).__init__()
        # Gri tonlamalı resim (1 kanal) girer
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        
        # Tam Bağlantılı Katmanlar
        self.fc1 = nn.Linear(64 * 32 * 32, 128)
        self.fc2 = nn.Linear(128, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1) # Düzleştir
        x = self.relu(self.fc1(x))
        x = self.sigmoid(self.fc2(x))
        return x

# ==========================================
# 3. EĞİTİM DÖNGÜSÜ
# ==========================================

# Veriyi Hazırla
dataset = FanDataset(DATA_PATH)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Modeli Başlat
model = FanModel().to(DEVICE)
criterion = nn.BCELoss() # Binary Cross Entropy (0 veya 1 için)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("\n🔥 Eğitim Başlıyor (-6dB Verisi ile)...\n")

loss_history = []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE).unsqueeze(1)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        # Doğruluk Hesabı
        predicted = (outputs > 0.5).float()
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(dataloader)
    accuracy = 100 * correct / total
    loss_history.append(epoch_loss)
    
    print(f"Epoch {epoch+1}/{EPOCHS} -> Loss: {epoch_loss:.4f} | Accuracy: %{accuracy:.2f}")

print("\n✅ Eğitim Tamamlandı!")

# ==========================================
# 4. KAYDETME
# ==========================================
SAVE_NAME = "fan_model_minus6db.pth"
torch.save(model.state_dict(), SAVE_NAME)
print(f"💾 Model kaydedildi: {SAVE_NAME}")

# İndirme Linki Oluştur (Kaggle İçin)
from IPython.display import FileLink
display(FileLink(SAVE_NAME))

🚀 Eğitim cuda üzerinde yapılacak.
✅ Toplam Dosya: 400
   - Normal: 200
   - Arıza: 200

🔥 Eğitim Başlıyor (-6dB Verisi ile)...

Epoch 1/20 -> Loss: 0.8019 | Accuracy: %50.00
Epoch 2/20 -> Loss: 0.6934 | Accuracy: %49.25
Epoch 3/20 -> Loss: 0.6922 | Accuracy: %53.25
Epoch 4/20 -> Loss: 0.6789 | Accuracy: %58.00
Epoch 5/20 -> Loss: 0.6798 | Accuracy: %55.00
Epoch 6/20 -> Loss: 0.6544 | Accuracy: %61.75
Epoch 7/20 -> Loss: 0.6025 | Accuracy: %64.25
Epoch 8/20 -> Loss: 0.5572 | Accuracy: %70.25
Epoch 9/20 -> Loss: 0.5063 | Accuracy: %72.75
Epoch 10/20 -> Loss: 0.4410 | Accuracy: %79.00
Epoch 11/20 -> Loss: 0.4379 | Accuracy: %79.50
Epoch 12/20 -> Loss: 0.3938 | Accuracy: %81.75
Epoch 13/20 -> Loss: 0.3368 | Accuracy: %84.75
Epoch 14/20 -> Loss: 0.2829 | Accuracy: %87.75
Epoch 15/20 -> Loss: 0.2413 | Accuracy: %89.75
Epoch 16/20 -> Loss: 0.1951 | Accuracy: %93.00
Epoch 17/20 -> Loss: 0.2033 | Accuracy: %91.00
Epoch 18/20 -> Loss: 0.1385 | Accuracy: %96.25
Epoch 19/20 -> Loss: 0.1176 | Accur

/kaggle/working/fan_model_minus6db.pth

In [6]:
import torch
import torch.nn as nn
import torchaudio
from torchaudio import transforms
import glob
import random
import os
from IPython.display import Audio, display

# ==========================================
# ⚙️ AYARLAR
# ==========================================
# Eğittiğin modelin dosya adı 
MODEL_PATH = "fan_model_minus6db.pth" 

# -6dB Veri Setinin Yeri
DATA_ROOT = "/kaggle/input/mimii-mini-dataset-100-nums-of-all/mimii-mini/-6_dB_fan/fan"

SAMPLE_RATE = 16000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"🚀 Test İşlemi {DEVICE} üzerinde yapılıyor...")

# ==========================================
# 1. MODEL MİMARİSİ 
# ==========================================
class FanModel(nn.Module):
    def __init__(self):
        super(FanModel, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.fc1 = nn.Linear(64 * 32 * 32, 128)
        self.fc2 = nn.Linear(128, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.sigmoid(self.fc2(x))
        return x

# ==========================================
# 2. ÖN İŞLEME (Preprocessing)
# ==========================================
def preprocess_audio(file_path):
    # Yükle
    waveform, sr = torchaudio.load(file_path)
    
    # KANAL SEÇİMİ (8 Kanal -> 1 Kanal)
    if waveform.shape[0] > 1:
        waveform = waveform[0:1, :] 
    
    # Resample
    if sr != SAMPLE_RATE:
        resampler = transforms.Resample(orig_freq=sr, new_freq=SAMPLE_RATE)
        waveform = resampler(waveform)
    
    # Kes/Doldur 
    num_samples = SAMPLE_RATE * 3
    if waveform.shape[1] > num_samples:
        waveform = waveform[:, :num_samples]
    else:
        padding = num_samples - waveform.shape[1]
        waveform = torch.nn.functional.pad(waveform, (0, padding))
        
    # GPU'ya al
    waveform = waveform.to(DEVICE)
    
    # Spektrogram
    mel_transform = transforms.MelSpectrogram(sample_rate=SAMPLE_RATE, n_mels=128, n_fft=1024, hop_length=512).to(DEVICE)
    spec = mel_transform(waveform)
    spec = transforms.AmplitudeToDB().to(DEVICE)(spec)
    
    # Boyutlandır (128x128)
    # Model [Batch, 1, 128, 128] bekler
    spec = spec.unsqueeze(0) # [1, 1, 128, Zaman]
    spec = torch.nn.functional.interpolate(spec, size=(128, 128), mode='bilinear', align_corners=False)
    spec = (spec - spec.min()) / (spec.max() - spec.min() + 1e-6)
    
    return spec

# ==========================================
# 3. TEST DÖNGÜSÜ
# ==========================================

# Modeli Yükle
if os.path.exists(MODEL_PATH):
    model = FanModel().to(DEVICE)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()
    print("✅ Model başarıyla yüklendi.\n")
else:
    print("❌ Model dosyası bulunamadı! Lütfen önce eğitimi çalıştır.")
    exit()

# Dosyaları Bul
normal_files = glob.glob(os.path.join(DATA_ROOT, "*", "normal", "*.wav"))
abnormal_files = glob.glob(os.path.join(DATA_ROOT, "*", "abnormal", "*.wav"))

# Rastgele Seçim (3 Normal, 3 Arızalı)
test_files = random.sample(normal_files, 3) + random.sample(abnormal_files, 3)
random.shuffle(test_files) # Sırayı karıştır

print(f"📊 TOPLAM {len(test_files)} DOSYA TEST EDİLİYOR...\n")
print(f"{'BEKLENEN':<10} | {'SKOR':<8} | {'SONUÇ':<10} | {'DOSYA'}")
print("-" * 60)

for file_path in test_files:
    # Gerçek durumu dosya yolundan anla
    is_abnormal = "abnormal" in file_path
    expected = "ARIZA" if is_abnormal else "NORMAL"
    
    # Tahmin Et
    input_tensor = preprocess_audio(file_path)
    with torch.no_grad():
        score = model(input_tensor).item()
    
    # Karar
    prediction = "ARIZA 🚨" if score > 0.5 else "NORMAL ✅"
    
    # Ekrana Yaz
    print(f"{expected:<10} | %{score*100:05.2f}  | {prediction:<10} | ...{os.path.basename(file_path)}")
    
    

🚀 Test İşlemi cuda üzerinde yapılıyor...
✅ Model başarıyla yüklendi.

📊 TOPLAM 6 DOSYA TEST EDİLİYOR...

BEKLENEN   | SKOR     | SONUÇ      | DOSYA
------------------------------------------------------------
ARIZA      | %99.97  | ARIZA 🚨    | ...00000025.wav
NORMAL     | %00.03  | NORMAL ✅   | ...00000041.wav
ARIZA      | %76.53  | ARIZA 🚨    | ...00000046.wav
NORMAL     | %01.71  | NORMAL ✅   | ...00000047.wav
ARIZA      | %83.43  | ARIZA 🚨    | ...00000032.wav
NORMAL     | %05.67  | NORMAL ✅   | ...00000042.wav
